In [1]:
# Lab8_llm.py requires to be loaded, hence this cell handles that

import glob
import shutil
import os

files = glob.glob("/kaggle/input/**/lab8_llm.py", recursive=True)

print("Found:", files)

shutil.copy(files[0], "/kaggle/working/lab8_llm.py")

print("File copied successfully!")

Found: ['/kaggle/input/datasets/alikhattak91/lab8-llm-py/lab8_llm.py']
File copied successfully!


In [2]:
# GUIDED PRACTISES 

In [3]:
# ACTIVITY 1: Connect the Model and Log Every Call

import json
import re
import statistics

import lab8_llm as llm
from lab8_llm import ask

llm.set_seed(0)
print(ask("Say hello in on short sentence."))
call = llm.LOG[-1]

print("calls so far:", len(llm.LOG), "| seconds:", call["seconds"], "| reply length:", len(call["reply"]))

Hello from the Lab 8 stand-in. I am a Python function, not a language model.
calls so far: 1 | seconds: 0.0 | reply length: 76


In [4]:
# Activity 2: Context Makes the Difference
QUESTION = "Why does the game crash when the player opens the inventory?"
CONTEXT = {
    "log" : "Error log: " + llm.LOG_TEXT.replace("\n", " "),
    "script": "Script line: 42: label.text = slot.item.name;",
    "version": "Engine version: Unity 2022.3 LTS"
}

def build_prompt(question, items):
    parts = [CONTEXT[name] for name in items]
    return "\n".join(parts + ["Question: " + question])


for items in [[], ["log"]]:
    prompt = build_prompt(QUESTION, items)
    reply = ask(prompt)
    print(f"context {items or 'none'}: {len(prompt)} characters sent")
    print("  reply:", reply)

context none: 70 characters sent
  reply: I cannot be sure without more information. Common causes of a crash when a menu opens are null references, missing assets or endless loops.
context ['log']: 223 characters sent
  reply: Likely cause: a NullReferenceException at InventoryUI.cs:42, so something on that line is empty.


In [5]:
# Activity 3: Decompose a goal and validate the plan 

def clean_json(text):
    text = text.strip()
    if text.startswith("```", 1):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return text.strip()

def check_plan(text):
    try:
        plan = json.loads(clean_json(text))
    except json.JSONDecodeError:
        return None, "not valid JSON"
    if not isinstance(plan, list) or not 3 <= len(plan) <= 6:
        return None, "needs 3 to 6 steps"
    if not all(isinstance(s, dict) and {"step", "check"} <= s.keys() for s in plan):
        return None, "a step is missing 'step' or 'check'"
    return plan, ""

def plan_prompt(goal):
    return (f"Goal: {goal}\n Split the goal into 3 and 6 steps. Reply with a JSON array only. "
            'Each item must look like {"step": "...", "check": "how to know it is done"}.')


for seed in (0,1):
    llm.set_seed(seed)
    plan, problem = check_plan(ask(plan_prompt("Create a simple enemy AI prototype")))
    print(f"seed {seed}: problem = {problem or 'none'}")
    for i, item in enumerate(plan or [], 1):
       print(f"{i}. {item['step']} -> check: {item['check']}")
            

seed 0: problem = none
1. Define behaviour -> check: Patrol, chase and retreat are written down
2. Choose states -> check: Each behaviour is a named state
3. Implement logic -> check: Events switch the enemy between states
4. Test scenarios -> check: Player distance and obstacles are tested
seed 1: problem = not valid JSON


In [6]:
# INDEPENDANT LAB TASKS

In [7]:
# LAB TASK 1 - ZERO SHOT OR FEW SHOT

REPORTS = [
    ("Player falls through the floor after a jump", "Physics / Collision"),
    ("Enemy walks through a wall", "Physics / Collision"),
    ("Game freezes for five seconds when loading level 3", "Performance"),
    ("Frame rate drops to 12 fps in the forest scene", "Performance"),
    ("Textures take a long time to appear", "Performance"),
    ("Boss health bar is not shown on the screen", "UI"),
    ("Inventory button does not respond when clicked", "UI"),
    ("Score resets to zero after pausing", "Game Logic"),
    ("Enemy keeps patrolling after the player is dead", "Game Logic"),
    ("Door stays closed even after the key is collected", "Game Logic"),
    ("Footstep sound plays twice for each step", "Audio"),
    ("Music stops when the menu is opened", "Audio"),
]

CATEGORIES = "Physics / Collision, Performance, UI, Game Logic, Audio"

ZERO = (
    "Classify this bug report as one of: " + CATEGORIES + ". "
    "Also give a severity (Low, Medium or High) and one action.\n"
    "Report: {report}"
)

FEW = (
    "Classify bug reports. Use exactly the format: Category | Severity | Action\n"
    "Input: Player falls through the floor\n"
    "Output: Physics / Collision | High | Check collider and collision layer\n"
    "Input: Loading screen freezes the game\n"
    "Output: Performance | Medium | Profile the scene and check asset sizes\n"
    "Input: {report}\nOutput:"
)

PATTERN = re.compile(
    r"^(Physics / Collision|Performance|UI|Game Logic|Audio) \| "
    r"(Low|Medium|High) \| (.+)$"
)

def parse(reply):
    m = PATTERN.match(reply.strip())
    return (m.group(1), m.group(2), m.group(3)) if m else None

def run_method(template, seed):
    llm.set_seed(seed)
    valid = correct = 0
    for report, truth in REPORTS:
        parsed = parse(ask(template.format(report = report)))
        if parsed:
            valid += 1
            correct += parsed[0] == truth

    return valid, correct

for name, template in [("zero-shot", ZERO), ("few-shot", FEW)]:
    result = [run_method(template, seed) for seed in range(10)]
    valid = sum(v for v, _ in result)
    correct = sum(c for _, c in result)
    total = 10 * len(REPORTS)
    print(f"{name}: format valid {valid}/{total} ((valid / total: .0%)),"
         f"correct among valid {correct}/{valid} ({correct / valid: .0%}), "
         f"correct overall {correct}/{total} ({correct / total: .0%})")

zero-shot: format valid 49/120 ((valid / total: .0%)),correct among valid 34/49 ( 69%), correct overall 34/120 ( 28%)
few-shot: format valid 115/120 ((valid / total: .0%)),correct among valid 107/115 ( 93%), correct overall 107/120 ( 89%)


In [8]:
# LAB TASK 2 - HOW MUCH CONTEXT IS ENOUGH

def score_answer(reply):
    # if 0 then no cause
    # if 1 then name the exception which occured
    # if 2,then name the expections and slot.item as well
    reply = reply.lower()
    if "nullreferenceexception" in reply and "slot.item" in reply: 
        return 2
    elif "nullreferenceexception" in reply:
        return 1
    else:
        return 0


for items in [[], ["log"], ["log", "script"], ["log", "script", "version"]]:
    prompt = build_prompt(QUESTION, items)
    llm.set_seed(0)
    # Ask the model first, then score the reply, and print one line in the required format
    reply = llm.ask(prompt)
    score = score_answer(reply)
    characters = len(prompt)

    context = items if items else "none"
    print(f"context {context}  characters {characters} score {score}")
    

context none  characters 70 score 0
context ['log']  characters 223 score 1
context ['log', 'script']  characters 269 score 2
context ['log', 'script', 'version']  characters 302 score 2


In [9]:
# LAB TASK 3 - PLAN, VALIDATE, RETRY
def get_plan(goal, max_retries = 2):
    
    # Return (plan, attempts_used) planis None if every attempt failed
    prompt = plan_prompt(goal)
    
    for attempt in range(max_retries + 1):
        # Ask the model
        reply = llm.ask(prompt)

        # Check whether the plan is valid 
        plan, reason = check_plan(reply)
        if plan is not None: 
            return plan, attempt + 1

        # Extend prompt with rejection reason
        prompt += f"\nYour last answer was rejected because: {reason}. Only return valid JSON."
        
    return None, max_retries + 1

# Experimenting

GOALS = ["Create a simple enemy AI prototype", 
         "Prepare a playable demo for Friday",
        "Add a save system to the game"]

first_try_ok = retries_used = failures = 0
llm.LOG.clear()

for seed in range(20):
    llm.set_seed(seed)
    
    for goal in GOALS:
        # Get plan and attempts
        plan, attempts = get_plan(goal)
        # Update counters
        if plan is None:
            failures += 1
        elif attempts == 1:
            first_try_ok += 1
        else:
            retries_used += attempts - 1

total = 20 * len(GOALS)

# Print results
print(f"Total plans:       {total}")
print(f"First try success: {first_try_ok}")
print(f"Retries used:      {retries_used}")
print(f"Failures:          {failures}")
print(f"Total LLM calls:   {len(llm.LOG)}")

Total plans:       60
First try success: 34
Retries used:      27
Failures:          3
Total LLM calls:   93


In [10]:

# Lab Task 4 - A ReAct Debugging Agent

# The tools and the parser (Given)

FILES = {
    "InventoryUI.cs": "40: foreach (var slot in slots)\n41: {\n42:          label.text = slot.item.name;\n43:  }"
}

TOOLS = {
    "read_log": lambda arg: llm.LOG_TEXT,
    "search_file": lambda arg: FILES.get(arg, "file not found: " + arg),
    "check_version": lambda arg: "Unity 2022.3 LTS"
}

HEADER = (
    "You are a debugging agent. Available tools: read_log[], search_file[name], check_version[].\n"
    "Reply with 'Thought: ...' and then either 'Action: tool[argument]' or 'Final: answer'.\n"
)

def parse_step(reply):
    # Return ('final', text) or ('action', tool, arg) or ('bad', reply).

    final = re.search(r"Final:\s*(.+)", reply, re.S)
    if final:
        return ("final", final.group(1).strip())

    act = re.search(r"Action:\s*(\w+)\[(.*?)\]", reply)
    if act:
        return ("action", act.group(1), act.group(2))

    return ("bad", reply)


# The Loop

def react(question, max_steps=5):
    transcript = HEADER + "Question: " + question + "\n"
    trace = []

    for step in range(1, max_steps + 1):

        # Ask the model
        reply = llm.ask(transcript)

        # Parse the response
        kind = parse_step(reply)

        # Case 1: A Final Answer
        if kind[0] == "final":
            answer = kind[1]
            trace.append(f"step {step}: Final -> {answer}")
            return answer, trace

        # Case 2: Bad reply
        elif kind[0] == "bad":
            observation = "Invalid format. Use Thought: followed by Action: or Final:"
            trace.append(f"step {step}: bad reply -> {observation}")

        # Case 3 and 4: Tool action
        else:
            tool = kind[1]
            arg = kind[2]

            if tool not in TOOLS:
                observation = (
                    f"unknown tool {tool}. Allowed: {', '.join(TOOLS.keys())}"
                )
            else:
                # Execute the allowed tool
                observation = TOOLS[tool](arg)

            trace.append(
                f"step {step}: {tool}[{arg}] -> "
                f"{observation.replace(chr(10), ' ')[:80]}"
            )

        # Add reply and observation to transcript
        transcript += reply + "\nObservation: " + observation + "\n"

    return None, trace + ["stopped: step limit reached"]


# Run the Experiment

QUESTION4 = "Why does the game crash when the inventory opens?"

llm.set_seed(2)
answer, trace = react(QUESTION4)

print("\n".join(trace))
print("answer:", answer)

outcomes = {"solved": 0, "step limit": 0}
unknown_tool_seen = 0

for seed in range(30):
    llm.set_seed(seed)
    llm.LOG.clear()

    answer, trace = react(QUESTION4)

    outcomes["solved" if answer else "step limit"] += 1
    unknown_tool_seen += any("unknown tool" in t for t in trace)

print(
    "30 normal runs:", outcomes,
    "| runs that tried an unknown tool:", unknown_tool_seen
)

llm.set_seed(0, mode="stuck")
answer, trace = react(QUESTION4)

print(
    "stuck model:", answer,
    "|", trace[-1],
    "| steps:", len(trace) - 1
)


step 1: read_log[] -> ERROR NullReferenceException: Object reference not set to an instance of an obje
step 2: search_file[InventoryUI.cs] -> 40: foreach (var slot in slots) 41: { 42:          label.text = slot.item.name; 
step 3: run_game[] -> unknown tool run_game. Allowed: read_log, search_file, check_version
step 4: run_game[] -> unknown tool run_game. Allowed: read_log, search_file, check_version
step 5: Final -> In InventoryUI.cs line 42, check that slot.item is not null before reading name.
answer: In InventoryUI.cs line 42, check that slot.item is not null before reading name.
30 normal runs: {'solved': 30, 'step limit': 0} | runs that tried an unknown tool: 3
stuck model: None | stopped: step limit reached | steps: 5


In [11]:
# Lab Task 05 - Catch a Hallucinated Function

# CHECK AGAINST A TRUSTED LIST

TRUSTED = {
    "Physics.Raycast",
    "transform.Translate",
    "Destroy",
    "audioSource.Play",
    "transform.Find"
}

TASKS = [
    "cast a ray forward from the player",
    "move the character with its speed",
    "destroy the object after two seconds",
    "play the sound attached to this object",
    "find the child object called Weapon"
]

def called_names(line):
    return re.findall(r"([A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*)\s*\(", line)


def unsupported(line):
    """Return the called names in the line that are not in TRUSTED."""
    names = called_names(line)
    return [name for name in names if name not in TRUSTED]


# First experiment
llm.set_seed(3)

for task in TASKS:
    line = llm.ask(f"Write one line of C# to {task}.")

    bad_names = unsupported(line)

    if bad_names:
        print(f"UNSUPPORTED {bad_names} | {line}")
    else:
        print(f"supported | {line}")


# Second experiment: 40 seeds
lines_checked = 0
flagged = 0
real_function_not_flagged = 0

for seed in range(40):
    llm.set_seed(seed)

    for task in TASKS:
        line = llm.ask(f"Write one line of C# to {task}.")

        lines_checked += 1

        bad_names = unsupported(line)

        if bad_names:
            flagged += 1

        if line.strip() == "Destroy(2f, gameObject);" and not bad_names:
            real_function_not_flagged += 1


# Calculate percentage
percentage = (flagged / lines_checked) * 100

# Print results
print(f"lines checked: {lines_checked}, flagged as unsupported: {flagged} ({percentage:.1f}%)")
print(f"real function used wrongly and NOT flagged: {real_function_not_flagged}")

UNSUPPORTED ['Physics.RaycastSmart'] | Physics.RaycastSmart(origin, direction);
supported | transform.Translate(move * speed * Time.deltaTime);
supported | Destroy(gameObject, 2f);
supported | audioSource.Play();
supported | transform.Find("Weapon");
lines checked: 200, flagged as unsupported: 54 (27.0%)
real function used wrongly and NOT flagged: 2
